# HW1: Banking Intent Classification

This notebook implements a simple baseline for **intent classification**.

Given a short customer-service query, the model predicts which banking intent it belongs to.

We will:

1. Download the BANKING77 CSV files directly from the official PolyAI GitHub repository.
2. Inspect the labels and examples.
3. Tokenize text with a simple whitespace-based tokenizer.
4. Build Bag-of-Words features.
5. Train a Multinomial Naive Bayes classifier.
6. Evaluate it on the test set.
7. Inspect common mistakes.


## 1. Install and import packages

In [1]:
!pip -q install scikit-learn pandas matplotlib

In [ ]:
import re
import json
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 2. Load the dataset

BANKING77 contains short customer-service queries from the banking domain.

Each example has:

- `text`: a customer query
- `category`: one of 77 banking intents

We download the CSV files directly from the official PolyAI GitHub repository.


In [ ]:
BASE_URL = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data"

train_url = f"{BASE_URL}/train.csv"
test_url = f"{BASE_URL}/test.csv"
categories_url = f"{BASE_URL}/categories.json"

train_df = pd.read_csv(train_url)
test_df = pd.read_csv(test_url)

# categories.json is convenient but not strictly required.
# We keep label names sorted for deterministic integer encoding.
label_names = sorted(train_df["category"].unique())

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Number of labels:", len(label_names))
print(label_names[:20])


Train shape: (10003, 2)
Test shape: (3080, 2)
Number of labels: 77
['Refund_not_showing_up', 'activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire', 'card_acceptance', 'card_arrival', 'card_delivery_estimate', 'card_linking', 'card_not_working', 'card_payment_fee_charged', 'card_payment_not_recognised', 'card_payment_wrong_exchange_rate', 'card_swallowed']


In [ ]:
train_df.head()


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


In [ ]:
test_df.head()


,text,category
0,How do I locate my card?,card_arrival
1,"I still have not received my new card, I order...",card_arrival
2,I ordered a card but it has not arrived. Help ...,card_arrival
3,Is there a way to know when my card will arrive?,card_arrival
4,My card has not arrived yet.,card_arrival


In [ ]:
print("Train size:", len(train_df))
print("Test size:", len(test_df))

print("\nTrain label distribution:")
print(train_df["category"].value_counts().head(10))

print("\nNumber of labels in train:", train_df["category"].nunique())
print("Number of labels in test:", test_df["category"].nunique())


Train size: 10003
Test size: 3080

Train label distribution:
category
card_payment_fee_charged                            187
direct_debit_payment_not_recognised                 182
balance_not_updated_after_cheque_or_cash_deposit    181
wrong_amount_of_cash_received                       180
cash_withdrawal_charge                              177
transaction_charged_twice                           175
declined_cash_withdrawal                            173
transfer_fee_charged                                172
balance_not_updated_after_bank_transfer             171
transfer_not_received_by_recipient                  171
Name: count, dtype: int64

Number of labels in train: 77
Number of labels in test: 77


## 3. Encode labels

Scikit-learn classifiers expect numerical labels, so we map each intent string to an integer.

We will still keep the readable label names for reports and error analysis.


In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(label_names)

train_df["label"] = label_encoder.transform(train_df["category"])
test_df["label"] = label_encoder.transform(test_df["category"])

# This list maps integer label IDs back to intent names.
label_names = list(label_encoder.classes_)

train_df.head()


,text,category,label
0,I am still waiting on my card?,card_arrival,12
1,What can I do if my card still hasn't arrived ...,card_arrival,12
2,I have been waiting over a week. Is the card s...,card_arrival,12
3,Can I track my card while it is in the process...,card_arrival,12
4,"How do I know if I will get my card, or if it ...",card_arrival,12


## 4. Inspect examples

Many labels are similar, so this task is not trivial.

For example, queries about card arrival, card activation, card linking, card not working, and lost cards can be easy to confuse.


In [ ]:
sample_df = train_df.sample(10, random_state=42)

for _, row in sample_df.iterrows():
    print("TEXT:", row["text"])
    print("LABEL:", row["category"])
    print()


TEXT: Is it possible for me to change my PIN number?
LABEL: change_pin

TEXT: I'm not sure why my card didn't work
LABEL: declined_card_payment

TEXT: I don't think my top up worked
LABEL: top_up_failed

TEXT: Can you explain why my payment was charged a fee?
LABEL: card_payment_fee_charged

TEXT: How long does a transfer from a UK account take? I just made one and it doesn't seem to be working, wondering if everything is okay
LABEL: balance_not_updated_after_bank_transfer

TEXT: Why am I getting declines when trying to make a purchase online?
LABEL: declined_transfer

TEXT: What is the $1 transaction on my account?
LABEL: extra_charge_on_statement

TEXT: It looks like my card payment was sent back.
LABEL: reverted_card_payment?

TEXT: Why am I unable to transfer money when I was able to before?
LABEL: beneficiary_not_allowed

TEXT: What if there is an error on the exchange rate?
LABEL: card_payment_wrong_exchange_rate



## 5. A simple whitespace tokenizer

For this baseline, we use a deliberately simple tokenizer:

1. lowercase the text;
2. separate punctuation from words;
3. split on whitespace.

This is not a perfect tokenizer, but it is enough for a basic Bag-of-Words Naive Bayes baseline.


In [ ]:
def simple_tokenize(text):
    text = str(text).lower()

    # Put spaces around punctuation so that "card?" becomes "card ?"
    text = re.sub(r"([.,!?;:()\[\]{}\"'])", r" \1 ", text)

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text.split()

examples = [
    "How do I activate my card?",
    "Why was I charged a fee?",
    "I can't make a bank transfer!",
    "Can I change my phone number?"
]

for text in examples:
    print("TEXT:", text)
    print("TOKENS:", simple_tokenize(text))
    print()


TEXT: How do I activate my card?
TOKENS: ['how', 'do', 'i', 'activate', 'my', 'card', '?']

TEXT: Why was I charged a fee?
TOKENS: ['why', 'was', 'i', 'charged', 'a', 'fee', '?']

TEXT: I can't make a bank transfer!
TOKENS: ['i', 'can', "'", 't', 'make', 'a', 'bank', 'transfer', '!']

TEXT: Can I change my phone number?
TOKENS: ['can', 'i', 'change', 'my', 'phone', 'number', '?']



## 6. Build Bag-of-Words features

`CountVectorizer` builds a vocabulary from the training set and represents each query as token counts.

For example:

```text
"how do I activate my card"
```

becomes a sparse vector counting tokens such as `how`, `activate`, `card`, and so on.


In [ ]:
vectorizer = CountVectorizer(
    tokenizer=simple_tokenize,
    token_pattern=None,   # required when using a custom tokenizer
    lowercase=False,      # we lowercase inside simple_tokenize
    min_df=1,
)

X_train = vectorizer.fit_transform(train_df["text"])
y_train = train_df["label"].values

X_test = vectorizer.transform(test_df["text"])
y_test = test_df["label"].values

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))


X_train shape: (10003, 2415)
X_test shape: (3080, 2415)
Vocabulary size: 2415


## 7. Train Multinomial Naive Bayes

Multinomial Naive Bayes is a classic baseline for text classification.

It estimates which tokens are likely to appear in each intent class.


In [ ]:
clf = MultinomialNB(alpha=1.0)
clf.fit(X_train, y_train)

MultinomialNB()

## 8. Evaluate on the full test set

In [ ]:
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")

Test accuracy: 0.7737


In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=label_names,
    digits=4,
    zero_division=0,
))


                                                  precision    recall  f1-score   support

                           Refund_not_showing_up     0.8571    0.9000    0.8780        40
                                activate_my_card     0.6250    0.8750    0.7292        40
                                       age_limit     1.0000    0.9500    0.9744        40
                         apple_pay_or_google_pay     0.8333    1.0000    0.9091        40
                                     atm_support     0.8929    0.6250    0.7353        40
                                automatic_top_up     0.8500    0.8500    0.8500        40
         balance_not_updated_after_bank_transfer     0.7250    0.7250    0.7250        40
balance_not_updated_after_cheque_or_cash_deposit     0.7255    0.9250    0.8132        40
                         beneficiary_not_allowed     0.7442    0.8000    0.7711        40
                                 cancel_transfer     0.7917    0.9500    0.8636        40
         

## 9. Inspect mistakes

Look at the examples the model gets wrong.

This is usually more useful than looking only at accuracy.


In [ ]:
test_result_df = test_df.copy()
test_result_df["pred"] = y_pred
test_result_df["pred_name"] = [label_names[i] for i in y_pred]
test_result_df["correct"] = test_result_df["label"] == test_result_df["pred"]

errors = test_result_df[~test_result_df["correct"]].copy()

print("Number of errors:", len(errors))
errors[["text", "category", "pred_name"]].head(20)


Number of errors: 697


,text,category,pred_name
0,How do I locate my card?,card_arrival,card_linking
2,I ordered a card but it has not arrived. Help ...,card_arrival,cash_withdrawal_not_recognised
5,When will I get my card?,card_arrival,card_about_to_expire
11,How long does a card delivery take?,card_arrival,card_delivery_estimate
22,How long should my new card take to arrive?,card_arrival,card_delivery_estimate
26,I think something went wrong with my card deli...,card_arrival,reverted_card_payment?
40,Why won't my card show up on the app?,card_linking,top_up_failed
71,Is there a way to make my old card usable with...,card_linking,card_payment_not_recognised
93,Is it a good time to exchange?,exchange_rate,beneficiary_not_allowed
97,Will I get a curreng foreign exchange rate?,exchange_rate,wrong_exchange_rate_for_cash_withdrawal


## 10. Show high-confidence mistakes

Naive Bayes can be confidently wrong. These examples are useful for discussion.


In [ ]:
probs = clf.predict_proba(X_test)
max_probs = probs.max(axis=1)

test_result_df["confidence"] = max_probs

high_conf_errors = (
    test_result_df[~test_result_df["correct"]]
    .sort_values("confidence", ascending=False)
)

high_conf_errors[["text", "category", "pred_name", "confidence"]].head(20)


,text,category,pred_name,confidence
1626,"I was double charged, and the second charge is...",pending_card_payment,transaction_charged_twice,0.999993
2637,how many transactions can i make with a dispos...,get_disposable_virtual_card,disposable_card_limits,0.999888
1420,It seems someone used my card! There are a few...,compromised_card,cash_withdrawal_not_recognised,0.998570
666,OMG! I'm trying to load my card and it wont t...,pending_top_up,top_up_failed,0.997306
2176,I have tried transferring money numerous times...,beneficiary_not_allowed,failed_transfer,0.993510
393,I have tried to use my card several times and ...,card_not_working,declined_card_payment,0.992618
2728,The app said I withdrew cash at an ATM and I d...,cash_withdrawal_not_recognised,wrong_amount_of_cash_received,0.990114
1004,"Hi, i don't know what's going on i've just pai...",top_up_reverted,Refund_not_showing_up,0.989259
980,What stores will take my credit card as payment?,card_acceptance,pending_card_payment,0.988892
2453,What type of deposits do you accept into my ac...,top_up_by_cash_or_cheque,supported_cards_and_currencies,0.986920


## 11. Try your own queries

In [ ]:
def predict_intent(text, top_k=5):
    x = vectorizer.transform([text])
    probs = clf.predict_proba(x)[0]
    top_idx = np.argsort(probs)[::-1][:top_k]

    return [(label_names[i], probs[i]) for i in top_idx]

custom_queries = [
    "How can I activate my new card?",
    "Why did my cash withdrawal fail?",
    "I want to change my phone number.",
    "Can I cancel a bank transfer?",
    "Where can I find my card PIN?",
]

for query in custom_queries:
    print("QUERY:", query)
    for label, prob in predict_intent(query):
        print(f"  {label:35s} {prob:.4f}")
    print()


QUERY: How can I activate my new card?
  activate_my_card                    0.9512
  card_linking                        0.0399
  card_about_to_expire                0.0045
  card_arrival                        0.0033
  change_pin                          0.0002

QUERY: Why did my cash withdrawal fail?
  cash_withdrawal_charge              0.4151
  pending_cash_withdrawal             0.2798
  declined_cash_withdrawal            0.2125
  cash_withdrawal_not_recognised      0.0678
  wrong_exchange_rate_for_cash_withdrawal 0.0108

QUERY: I want to change my phone number.
  change_pin                          0.5510
  edit_personal_details               0.4201
  cancel_transfer                     0.0143
  lost_or_stolen_phone                0.0078
  terminate_account                   0.0016

QUERY: Can I cancel a bank transfer?
  cancel_transfer                     0.5521
  transfer_into_account               0.1922
  balance_not_updated_after_bank_transfer 0.1387
  failed_transfer     

## 12. Assignment

위의 베이스라인은 whitespace tokenizer + bag-of-words 기반 Naive Bayes classifier의 조합으로 77.37% 의 accuracy를 달성했습니다. 이제 다양한 부분을 tune 하셔서 더 높은 accuracy를 달성할 수 있도록 새 분류기를 학습시켜서 제출해주시면 됩니다.

가능한 tweak의 예시는 다음과 같습니다 (물론 exhaustive list가 아니고, 많은 다른 창의적인 접근이 있을 수 있습니다):

1. Tokenizer 교체. 예컨대 stopwords를 버리거나, 이미 잘 학습된 모델의 알려진 tokenizer를 가져다 쓸 수 있습니다.

2. Vocabulary 크기 제한. 제한된 크기의 training set에서는 오히려 feature 수를 줄이는 것이 generalization에 더 유리할 수 있습니다.

3. Smoothing 세기 조절. Baseline에서는 Laplace smoothing (alpha=1) 을 적용했지만, 더 유리한 정도가 있을 수 있습니다.

4. Feature representation 수정. Bag-of-words 에서 출현 횟수 대신 여부만 고려하거나, 순서를 고려할 수 있도록 n-gram 언어 모델을 사용하거나, character n-gram을 사용하는 것도 가능합니다.

11장까지의 예시 부분은 수정하지 마시고, 아래 내용을 수정하여 새로운 분류기를 구현 후, 마지막에 같은 방식으로 full test set에 대한 성능을 보여주시면 됩니다. 훈련 데이터 등은 반드시 그대로 사용하셔야 합니다 (외부 데이터, LLM 등 액세스는 금하는 것으로 하겠습니다).

### 채점 기준
- 성능 60%
  * 가장 높은 accuracy = 가장 낮은 error rate을 달성하신 수강생은 성능 점수 100 점을 받습니다.
  * 다른 수강생 분들은 해당 성능 대비 error rate이 2배 높을 때마다 20% 감점된 점수를 적용할 예정입니다.
  * 예시1) 최고 성능이 90% acc (10% error rate) 라면 80% acc (20% error rate) 를 달성한 수강생의 점수는 80 점이 됩니다.
  * 예시2) 최고 성능이 95% acc (5% error rate) 라면 90% acc 의 점수는 80 점, 80% acc 의 점수는 60 점이 됩니다.
  * 단 최고 성능이 95% acc 보다 좋다면 95% 를 넘긴 모든 수강생은 100 점으로 하고, 다른 수강생들의 점수는 5% erro rate을 기준으로 산정하겠습니다.
- 재현성 20%
  * 제출하신 노트북이 그대로 처음부터 끝까지 실행 가능한지
  * 리포트된 accuracy가 재현되는지
- 분석 20%
  * 어떤 점을 어떻게, 왜 수정했는지에 대한 간단한 설명
  * Error analysis 등 간단한 (비교)분석

### 제출
- https://ldilab.github.io/course-login/?course=samsung-ds-2026 로 접속하신 후, 개인 gmail로 인증하신 뒤 완성된 노트북 (.ipynb) 파일을 제출해주시면 됩니다.
- 기타 문의사항 등은 [ta.training@ldi.snu.ac.kr](mailto:ta.training@ldi.snu.ac.kr) 로 이메일 부탁드립니다.
- 제출 기한: 월요일 수업 시작 전까지 (6/29 9:30 AM)

In [ ]:
# Build your own classifier!
clf = ...
# clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")

print(classification_report(
    y_test,
    y_pred,
    target_names=label_names,
    digits=4,
    zero_division=0,
))